In [1]:
import os
import argparse
import numpy as np
import torch
import rbo as rbo_
from scipy.stats import rankdata
import resource
from pathlib import Path
from scipy.stats import ConstantInputWarning
from sklearn.metrics import ndcg_score
from sklearn.metrics import recall_score
import pandas as pd
import warnings
from sklearn.metrics import f1_score
import torch.nn.functional as F
from scipy import stats
from metrics import AlignmentMetrics
from representations.common_args import add_common_args
from importlib.machinery import SourceFileLoader
from tqdm import tqdm
import itertools

from structural_similarity import experiments_rep, experiments_func, load_results, build_inner_args

In [10]:

from schedule_representations import experiments, scripts_gpu, scripts_no_gpu

scripts = scripts_no_gpu + scripts_gpu

for exp in experiments:
    for script_a, script_b in itertools.combinations(scripts, 2):
        # Convert script paths to module-style names
        module_a = script_a.replace(".py", "").replace("/", ".")
        module_b = script_b.replace(".py", "").replace("/", ".")

        # Build readable job name
        label_a = script_a.split("/")[1].replace("run_", "").replace(".py", "")
        label_b = script_b.split("/")[1].replace("run_", "").replace(".py", "")
        job_name = f"[EEMC] Structural Similarity {label_a} x {label_b} for {exp['name']}"

        # Combine arguments
        args = [
            "--model", "./models/Qwen3-0.6B-Base_ftrace_date-lm_conversational_lr0.0001_seed42",
            "--train_dataset_name", exp["dataset"],
            "--train_dataset_split", "test",
            "--test_dataset_name", exp["dataset"],
            "--test_dataset_split", "test",
            # "--sim_script_a", module_a,
            # "--sim_script_b", module_b,
            # "--experiment_name", exp["name"]
        ]
        print(args)
        
        features_a, influence_estimate_a = load_results("representations.run_bm25", args)
        features_b, influence_estimate_b = load_results("representations.run_bm25", args)
        features_a = features_a - features_a.mean(dim=0, keepdim=True)
        features_b = features_b - features_b.mean(dim=0, keepdim=True)
        if False:
            features_a = F.normalize(features_a, dim=-1)
            features_b = F.normalize(features_b, dim=-1)
        
        break
    
    break

[INFO] HTTP Request: HEAD https://huggingface.co/datasets/loris3/ftrace_date-lm_conversational-sample/resolve/main/README.md "HTTP/1.1 200 OK"


['--model', './models/Qwen3-0.6B-Base_ftrace_date-lm_conversational_lr0.0001_seed42', '--train_dataset_name', 'loris3/ftrace_date-lm_conversational-sample', '--train_dataset_split', 'test', '--test_dataset_name', 'loris3/ftrace_date-lm_conversational-sample', '--test_dataset_split', 'test']


[INFO] HTTP Request: HEAD https://huggingface.co/datasets/loris3/ftrace_date-lm_conversational-sample/resolve/94fc648985b8cf2b42c597a8bcb9f4eb1b22951b/ftrace_date-lm_conversational-sample.py "HTTP/1.1 404 Not Found"
[INFO] HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/loris3/ftrace_date-lm_conversational-sample/loris3/ftrace_date-lm_conversational-sample.py "HTTP/1.1 404 Not Found"
[INFO] HTTP Request: HEAD https://huggingface.co/datasets/loris3/ftrace_date-lm_conversational-sample/resolve/94fc648985b8cf2b42c597a8bcb9f4eb1b22951b/.huggingface.yaml "HTTP/1.1 404 Not Found"
[INFO] HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=loris3/ftrace_date-lm_conversational-sample "HTTP/1.1 200 OK"
[INFO] HTTP Request: HEAD https://huggingface.co/datasets/loris3/ftrace_date-lm_conversational-sample/resolve/94fc648985b8cf2b42c597a8bcb9f4eb1b22951b/dataset_infos.json "HTTP/1.1 404 Not Found"
[INFO] HTTP Request: HEAD https://huggingface.c

adapter_path ./models/Qwen3-0.6B-Base_ftrace_date-lm_conversational_lr0.0001_seed42/adapter_config.json


[INFO] HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-0.6B-Base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-0.6B-Base/da87bfb608c14b7cf20ba1ce41287e8de496c0cd/config.json "HTTP/1.1 200 OK"
[INFO] HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-0.6B-Base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-0.6B-Base/da87bfb608c14b7cf20ba1ce41287e8de496c0cd/config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

... (5 outputs left out)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

[INFO] HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-0.6B-Base/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
[INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-0.6B-Base/da87bfb608c14b7cf20ba1ce41287e8de496c0cd/generation_config.json "HTTP/1.1 200 OK"


Tokenizer max length: 4096
influence_estimate_path: ./results/influence/BM25Estimator/-/ftrace_date-lm_conversational-sample_test/ftrace_date-lm_conversational-sample_test/estimate.parquet
dirname: ./results/influence/BM25Estimator/-/ftrace_date-lm_conversational-sample_test/ftrace_date-lm_conversational-sample_test
exists: True


Computing bm25 embeddings: 100%|██████████| 1000/1000 [00:00<00:00, 2061.16it/s]


In [11]:
features_a

tensor([[214.5896,   9.0436,  14.9941,  ...,   9.8696,   4.2847,   8.4902],
        [ -4.1922, 178.6763,  -4.9005,  ...,  -1.3495,  -5.0644,  -5.0414],
        [ 24.1475,  18.3809, 259.3503,  ...,  20.1508,  14.5797,  14.0968],
        ...,
        [-17.2446, -18.6839, -17.0492,  ...,  73.7479, -16.1248, -16.4741],
        [ -9.8796, -15.1926, -15.9517,  ..., -13.1065, 219.8980, -17.2885],
        [-13.2572, -12.5081, -15.4973,  ...,  -6.9906, -12.9554,  91.0046]])

In [12]:
for score_name, fn in tqdm(experiments_rep, desc="experiments_rep"):
    print(score_name)
    cka = fn

experiments_rep: 100%|██████████| 4/4 [00:00<00:00, 34952.53it/s]

cka
unbiased_cka
cknna
cycle_knn


## Is representaitonal/functional alignment consistent across different models trained on teh same data?